# 1. 包装式中间件 

- warp_model_call
    - dynamic_model
- warp_tool_call

## 1.1 API （Application Programming Interface）

- 装饰器

## 1.2 ModelRequest

```python
    """Model request information for the agent.

    Type Parameters:
        ContextT: The type of the runtime context. Defaults to `None` if not specified.
    """

    model: BaseChatModel                                            #换模型
    messages: list[AnyMessage]  # excluding system message          #换系统提示词：dynamic_prompt
    system_message: SystemMessage | None                           
    tool_choice: Any | None                                         #选择工具
    tools: list[BaseTool | dict[str, Any]]                          #动态加载
    response_format: ResponseFormat[Any] | None                     #动态指定结构化输出
    state: AgentState[Any]
    runtime: Runtime[ContextT]
    model_settings: dict[str, Any] = field(default_factory=dict)
```

# 2. wrap_model_call中间的应用

## 2.1 ModelRequest与ModelResponse的数据结构

In [ ]:
#引入模块
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from typing import Callable
from dataclasses import dataclass, fields

- override的作用：
    - 对dataclass操作
        - 直接操作（修改，读取）
    - 需要对dataclass数据保护
        - override克隆一份，进行修改，返回克隆的修改的备份，对原数据不进行任何修改

In [ ]:
from collections.abc import Callable
from typing import Callable


from langchain.agents.middleware.types import ExtendedModelResponse, ModelRequest, ModelResponse
from langchain_core.messages import AIMessage

class MyMiddleware(AgentMiddleware):
    """测试ModelRequest,ModelResponse,实现系统提示词增加指令
    """
    def __init__(self,extra_instruction:str=""):
        self.extra_instruction=extra_instruction

    def wrap_model_call(self, request, handler,) :
        
        #1. 观察modelRequest的数据结构(观察数据类有多少成员)
        # for f in fields(request):   #field(name,description,default)
        #     print(f"字段:{f.name}","\t\t",type(getattr(request,f.name)),"\t\t",getattr(request,f.name))
        if request.system_message:
            #提示词进行修改
            system_prompt = request.system_message.content
            system_prompt = f"{system_prompt}\n\n额外指令：{self,self.extra_instruction}"
            # 修改
            



        #1.1 动态修改系统提示词

        #2. 调用handler进行模型调用
        response = handler(request)

        #3. 观察ModelResponse
        for f in fields(response):
            print(f"字段：{f.name}","\t\t",type(getattr(response,f.name)))

        #3.1 修改ModelResponse

        #返回ModelResponse

        return response

In [5]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="ollama:qwen3.6:latest",
    base_url="http://192.168.8.21:11434"
)

agent=create_agent(
    model=model,
    middleware=[MyMiddleware()],
    system_prompt="你是Ai助手"
)

response = agent.invoke({
    "messages":[
        {"role":"user",
        "content":"是什么是修饰器？"}
    ]
})

字段:model <class 'langchain_ollama.chat_models.ChatOllama'> model='qwen3.6:latest' base_url='http://192.168.8.21:11434'
字段:messages <class 'list'> [HumanMessage(content='是什么是修饰器？', additional_kwargs={}, response_metadata={}, id='3fa1d815-98d1-421e-88be-e0719373369c')]
字段:system_message <class 'langchain_core.messages.system.SystemMessage'> content='你是Ai助手' additional_kwargs={} response_metadata={}
字段:tool_choice <class 'NoneType'> None
字段:tools <class 'list'> []
字段:response_format <class 'NoneType'> None
字段:state <class 'dict'> {'messages': [HumanMessage(content='是什么是修饰器？', additional_kwargs={}, response_metadata={}, id='3fa1d815-98d1-421e-88be-e0719373369c')]}
字段:runtime <class 'langgraph.runtime.Runtime'> Runtime(context=None, store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x12c3cefc0>, previous=None, execution_info=ExecutionInfo(checkpoint_id='1f142a82-31f9-6004-8000-0dfe3ea4ac87', checkpoint_ns='model:346d8908-2130-363c-b638-9cbd387aaf5d', task_id='346d8

In [6]:
from langchain.tools import tool
from langchain.agents.middleware import wrap_tool_call
from langchain.agents.middleware import ToolCallRequest
from langchain.messages import ToolMessage
from langgraph.types import Command
from typing import Callable, Any

@tool 
def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    print(">>> 工具收到的城市参数:", city)
    return f"{city}的天气不错，下雪中，温度零下10度"

@wrap_tool_call
def update_middleware(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command[Any]],
) -> ToolMessage | Command[Any]:

    # 修改参数：在原城市名后追加“市这个城市”
    original_city = request.tool_call["args"]["city"]
    modified_city = original_city + "市这个城市"
    
    modified_call = {
        **request.tool_call,
        "args": {
            **request.tool_call["args"],
            "city": modified_city
        },
    }
    
    print(f"中间件：原始参数 -> {original_city}")
    print(f"中间件：修改后参数 -> {modified_city}")
    
    # 创建新的请求（拷贝并覆盖）
    new_request = request.override(tool_call=modified_call)
    
    # 调用真正的工具
    response = handler(new_request)
    
    # 可选：也可以在此修改工具返回的内容
    return response

from langchain.chat_models import init_chat_model
model = init_chat_model(
    model="ollama:qwen3.6:latest",   
    base_url="http://192.168.8.21:11434"
)
# 创建 Agent，配置中间件
from langchain.agents import create_agent
agent = create_agent(
    model=model,
    tools=[get_weather],
    middleware=[update_middleware],
    system_prompt="你是一个有用的助手，必须使用 get_weather 工具查询天气，不要直接回答。"
)

# 运行
from langchain.messages import HumanMessage
result = agent.invoke({
    "messages": [HumanMessage("北京天气如何？")]
})

# 输出最终消息
final_message = result['messages'][-1]
print(f"最终回答: {final_message.content}")

中间件：原始参数 -> 北京
中间件：修改后参数 -> 北京市这个城市
>>> 工具收到的城市参数: 北京市这个城市
最终回答: 北京现在正在下雪，气温为零下10度。请注意保暖。


In [ ]:
# 多次调用handler
weather_call_count = 0

@tool
def get_weather(city:str) -> str :
    """查询指定城市的的天气信息"""
    global weather_call_count
    weather_call_count += 1
    if weather_call_count <= 2  #模拟一个异常
        raise Exception(f"API调用失败，第{weather_call_count}次调用，链接超时！")
    return f"{city}的天气：晴朗，温度20度"


class MultiCallMiddleware(AgentMiddleware):
    def __init__(self)

